# Kasese health-centre micro-grid: dispatch analysis

Two coupled demand constraints define how much energy must come from solar (x) and batteries (y) each day:

$$3x + 2y = D_1 \quad\text{(daytime load)}\qquad 4x + y = D_2 \quad\text{(critical-equipment load)}$$

All reusable logic lives in `src/`; this notebook only orchestrates it and shows results.

In [23]:
%matplotlib inline
import numpy as np
from src.config import CSV_PATH, PLOT_USAGE_PATH, PLOT_SENS_PATH, SOLAR_COST_UGX, BATTERY_COST_UGX
from src.grid import MicroGrid, HybridMicroGrid
from src.data import generate_demand_csv, load_demand_csv, demand_matrix, interactive_single_day
from src.solvers import time_solvers, handle_infeasible
from src.analysis import usage_stats, cost_model, sensitivity_analysis
from src.plotting import plot_usage_and_cost, plot_sensitivity

## 1. Well-posedness of the 2x2 system

In [ ]:
grid = MicroGrid()
det, cond = grid.diagnostics()

## 2. Demand data

Mode (b): generate 30 days of seeded demand data with a weekly pattern, save to CSV, reload.
Mode (a), interactive single-day entry, is available via `interactive_single_day()` (left commented out so the notebook runs top to bottom).

In [ ]:
df = generate_demand_csv(CSV_PATH, n_days=30)
df = load_demand_csv(CSV_PATH)
D = demand_matrix(df)
df.head(9)

# Interactive mode (uncomment to use):
# d1, d2 = interactive_single_day()
# print(grid.solve_day(d1, d2))

## 3. Solve all 30 days: loop vs vectorised

In [ ]:
X_loop = grid.solve_batch_loop(D)
X_vec = grid.solve_batch_vectorised(D)
assert np.allclose(X_loop, X_vec), "loop and vectorised results disagree!"

t_loop, t_vec = time_solvers(grid, D, number=300)
print(f"Loop version:       {t_loop * 1e6:8.2f} microseconds/call (30 separate solves)")
print(f"Vectorised version: {t_vec * 1e6:8.2f} microseconds/call (one solve, 30 columns)")
print(f"Speed-up: {t_loop / t_vec:.1f}x")

Both call the same LAPACK routine. The loop pays Python-call, array-allocation and factorisation overhead 30 times; the vectorised call factorises A once and back-substitutes for all 30 right-hand sides. It wins on overhead, not FLOPs (the matrix is tiny).

## 4. Infeasibility detection and handling

In [ ]:
X_clean, report = handle_infeasible(grid, X_vec, D)
n_bad = int(report["infeasible"].sum())
print(f"{n_bad} of {D.shape[1]} days produced a negative x or y from the raw linear solve.")
report[report["infeasible"]]

Flagged days are re-solved with non-negative least squares (`scipy.optimize.nnls`), which finds the closest physically valid dispatch (x, y >= 0) rather than silently clipping, which would break the original equations. Everything downstream uses `X_clean`.

## 5. Descriptive statistics

In [ ]:
stats = usage_stats(X_clean)
for k, v in stats.items():
    print(f"{k:8s}: mean={v['mean']:.2f} kWh, variance={v['variance']:.2f}, stdev={v['stdev']:.2f}")
more_volatile = "battery" if stats["battery"]["stdev"] > stats["solar"]["stdev"] else "solar"
print(f"-> {more_volatile} usage is more volatile (higher standard deviation).")

## 6. Cost model

In [ ]:
daily_cost, monthly_cost = cost_model(X_clean)
print(f"Rates: solar {SOLAR_COST_UGX} UGX/kWh, battery {BATTERY_COST_UGX} UGX/kWh")
print("First 5 days' cost (UGX):", np.round(daily_cost[:5], 1))
print(f"Total cost over 30 days: UGX {monthly_cost:,.0f}")
print(f"Average daily cost:      UGX {monthly_cost/30:,.0f}")

## 7. Usage and cost plot

In [ ]:
fig = plot_usage_and_cost(X_clean, daily_cost, PLOT_USAGE_PATH)

## Extension 1: hybrid system with a diesel generator (3x3)

In [ ]:
hybrid = HybridMicroGrid()
det3, cond3 = hybrid.diagnostics()
x, y, z = hybrid.solve_day(42.0, 25.0, 25.0)
print(f"Example day (D1=42, D2=25, D3=25) -> x={x:.2f}, y={y:.2f}, z={z:.2f} kWh")

In [ ]:
# Linearly dependent third equation
hybrid_singular = HybridMicroGrid(singular_demo=True)
hybrid_singular.diagnostics()
try:
    hybrid_singular.solve_day(42.0, 25.0, 25.0)
except np.linalg.LinAlgError as e:
    print(f"solve_day raised as expected: {e}")

When the third equation is a linear combination of the first two, it adds no new information. The matrix is singular (det ~ 0), so `scipy.linalg.solve` raises rather than guessing. Physically, the third constraint was redundant and cannot separate diesel from solar/battery: infinitely many (x, y, z) satisfy all three equations, or none do if D3 is inconsistent with D1, D2.

## Extension 2: Monte Carlo sensitivity (+-5% on D1, D2)

In [ ]:
d1_ex, d2_ex = 42.0, 25.0
X_mc, x_base, y_base = sensitivity_analysis(grid, d1_ex, d2_ex, pct=0.05, n_draws=1000)
x_std, y_std = X_mc[0].std(), X_mc[1].std()

print(f"Base solution at D1={d1_ex}, D2={d2_ex}: x={x_base:.3f}, y={y_base:.3f}")
print("After 1000 draws of +-5% noise on D1, D2:")
print(f"  x: mean={X_mc[0].mean():.3f}, std={x_std:.3f}  (relative spread {x_std/x_base*100:.1f}% of x)")
print(f"  y: mean={X_mc[1].mean():.3f}, std={y_std:.3f}  (relative spread {y_std/y_base*100:.1f}% of y)")
print(f"Condition number of A: {cond:.2f}")
print(f"Worst-case bound on relative swing: ~{cond*5:.0f}%")

fig = plot_sensitivity(X_mc, x_base, y_base, cond, PLOT_SENS_PATH)

The relative error in the solution is bounded by roughly cond(A) times the relative input error, so a 5% input perturbation could in the worst case swing x, y by cond(A) x 5%. The observed spread is smaller because random noise on two independent inputs rarely aligns with the single worst-case direction.